# 01 Data Cleaning\n\nGoal: load the raw headline file, check basic data quality, remove exact duplicates, and create daily next-day return targets.

In [ ]:
from pathlib import Path\nimport pandas as pd\n\nPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()\nRAW_PATH = PROJECT_ROOT / "data" / "raw" / "sp500_headlines_2008_2024.csv"\nPROCESSED_DIR = PROJECT_ROOT / "data" / "processed"\nPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(RAW_PATH, parse_dates=["Date"])\nprint(f"Raw rows: {len(df):,}")\nprint(f"Columns: {list(df.columns)}")\nprint(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")\nprint(f"Missing values:\n{df.isna().sum()}")\nprint(f"Exact duplicate rows: {df.duplicated().sum():,}")\ndf.head()

In [ ]:
cp_values_per_date = df.groupby("Date")["CP"].nunique()\nprint(f"Dates with multiple CP values: {(cp_values_per_date > 1).sum()}")\n\ndf_clean = df.drop_duplicates().sort_values(["Date", "Title"]).reset_index(drop=True)\ndaily = (\n    df_clean.groupby("Date")\n    .agg(\n        CP=("CP", "first"),\n        headline_count=("Title", "size"),\n        unique_headline_count=("Title", "nunique"),\n        daily_text=("Title", lambda x: " . ".join(x.astype(str))),\n    )\n    .reset_index()\n    .sort_values("Date")\n)\ndaily["return_next_day"] = daily["CP"].shift(-1) / daily["CP"] - 1\ndaily["direction_next_day"] = (daily["return_next_day"] > 0).astype(int)\ndaily = daily.dropna(subset=["return_next_day"]).reset_index(drop=True)\n\ndf_clean.to_csv(PROCESSED_DIR / "headlines_clean.csv", index=False)\ndaily.to_csv(PROCESSED_DIR / "daily_dataset.csv", index=False)\n\nprint(f"Clean headline rows: {len(df_clean):,}")\nprint(f"Daily rows: {len(daily):,}")\ndaily.head()